# GA4 Marketing Intelligence Autopilot: AI brief run

Run the cells **top to bottom, one at a time**. Each cell ends with an **OK** or **PROBLEM** line.
If you see PROBLEM, stop and send that output. Everything here is free: no card anywhere.

**Before you start (one time):**
1. Go to **build.nvidia.com**, click **Sign in / Join** (free NVIDIA Developer Program, no card).
2. Open **build.nvidia.com/settings/api-keys**, click **Generate API Key**, copy it (starts with `nvapi-`).
3. In Colab, click the **key icon** on the left, then **Add new secret**:
   - `NVIDIA_API_KEY` = your nvapi key, **Notebook access ON**
   - `GITHUB_TOKEN` = your GitHub token, **Notebook access ON**
4. Have `ga4-mia-v5.zip` downloaded on your laptop.

No GPU needed. The runtime can stay on CPU.

## 1. Setup: get your repo and the v5 code

In [ ]:
%cd /content
!rm -rf ga4-marketing-intelligence-autopilot ga4-mia-v5.zip
!git clone -q https://github.com/DevinderKaurLabs/ga4-marketing-intelligence-autopilot.git
from google.colab import files
print("Choose ga4-mia-v5.zip in the file picker:")
files.upload()
!unzip -qo ga4-mia-v5.zip
%cd /content/ga4-marketing-intelligence-autopilot
!pip install -q openai weasyprint python-dotenv jinja2 pydantic
!rm -rf artifacts/briefs artifacts/reports artifacts/eval

from pathlib import Path
n = len(list(Path("artifacts/payloads").glob("week_*.json")))
print("OK: setup done, 12 weekly payloads found" if n == 12 else f"PROBLEM: expected 12 payloads, found {n}")

## 2. Connect, pick models automatically, test each one
`PROVIDER = "nvidia"` gives Mistral, Qwen and NVIDIA Nemotron.
If NVIDIA ever fails, change it to `"gemini"` and add a `GEMINI_API_KEY` secret from aistudio.google.com.

In [ ]:
PROVIDER = "nvidia"   # or "gemini"

import os, sys
from google.colab import userdata
os.environ["LLM_PROVIDER"] = PROVIDER
os.environ["LLM_API_KEY"] = userdata.get("NVIDIA_API_KEY" if PROVIDER == "nvidia" else "GEMINI_API_KEY")
sys.path.insert(0, "src")

!python tests/test_ai_layer.py | tail -1

from mia import llm
candidates = llm.pick_models_for(PROVIDER, llm.list_model_ids())
print("Candidates:", candidates)
MODELS = []
for m in candidates:
    try:
        reply = llm.ping(m)
        print(f"  {m}: {reply}")
        MODELS.append(m)
    except Exception as e:
        print(f"  {m}: skipped ({str(e)[:120]})")
MODEL_A = MODELS[0] if MODELS else None
print("\nUsing:", MODELS)
print("OK: models connected" if MODELS else "PROBLEM: no model answered, send this output")

## 3. Smoke test: Christmas week, every model (about 1 to 3 minutes)
You want `validated`. `fallback` means the draft failed the checks; the lines under it show why.

In [ ]:
for m in MODELS:
    !python scripts/generate_briefs.py --model "{m}" --weeks 2020-12-21

## 4. All 12 weeks, all models, with evaluation (about 10 to 20 minutes)
`rate limited, waiting...` is the free tier pacing requests. Leave it running.

In [ ]:
MODELS_ARG = " ".join(f'"{m}"' for m in MODELS)
!python scripts/evaluate_models.py --models {MODELS_ARG} --weeks all

## 5. Pick the winner automatically and make PDFs (no model calls)

In [ ]:
import pandas as pd
runs = pd.read_csv("artifacts/eval/model_eval_runs.csv")
score = (runs.assign(valid=runs["status"].eq("validated"))
             .groupby("model")
             .agg(validated=("valid", "mean"), invented=("invented_numbers_first_try", "mean"),
                  weeks=("week_start", "count")))
score = score.sort_values(["validated", "invented"], ascending=[False, True])
print(score)
MODEL_PRIMARY = score.index[0]
Path("artifacts/PRIMARY_MODEL.txt").write_text(MODEL_PRIMARY)
print("\nWinner:", MODEL_PRIMARY)
!python scripts/render_pdfs.py --model "{MODEL_PRIMARY}"
print("OK: winner chosen and PDFs written" if score.iloc[0]["validated"] > 0 else "PROBLEM: no validated briefs, send this output")

## 6. Read a brief (change the date to read other weeks: 2020-11-23, 2021-01-25)

In [ ]:
from IPython.display import HTML
WEEK = "2020-12-21"
slug = MODEL_PRIMARY.replace("/", "_").replace(":", "_")
HTML(open(f"artifacts/reports/{slug}/brief_{WEEK}.html").read())

## 7. Download all briefs and the evaluation to your laptop

In [ ]:
!zip -qr briefs.zip artifacts/reports artifacts/eval
files.download("briefs.zip")
print("OK: download started")

## 8. Save everything to GitHub

In [ ]:
t = userdata.get("GITHUB_TOKEN")
!git config user.email "devinderkaur2191@gmail.com" && git config user.name "Devinder"
!git add -A && git commit -qm "AI layer: Mistral vs Qwen vs Nemotron briefs, evaluation, PDFs" && echo "committed"
!GIT_TERMINAL_PROMPT=0 git push https://{t}@github.com/DevinderKaurLabs/ga4-marketing-intelligence-autopilot.git main 2>&1 | sed "s/{t}/***/g" | tail -2

## Troubleshooting
- **Cell 2 error mentioning the key name**: the secret name must match exactly, with Notebook access ON.
- **A model says `skipped`**: fine, the others carry on. If all are skipped, switch `PROVIDER` to `"gemini"`.
- **`rate limited` messages**: normal on free tiers; the code waits and retries by itself.
- **Credits used up (NVIDIA)**: requests stop with an error; nothing is ever charged.
- **Anything else**: send the full output of the cell that failed.